<a href="https://colab.research.google.com/github/tardigrade-dot/colab-script/blob/main/02_voice_postprocess_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
音频拼接 & SRT 合并工具
功能：
  - 扫描目录下所有 wav/srt 文件对
  - 按文件名中 -数字1_数字2 排序
  - 将 wav 拼接为单个大音频（允许超出 5 小时）
  - 合并 srt 时间戳，序号重排
"""

import os
import re
import subprocess
import logging
import sys
from pathlib import Path
from datetime import timedelta

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger()

# ============ 配置 ============
MAX_HOURS = 5  # 超过此时长后切分为下一个文件
# ==============================


def get_sorted_wav_files(input_dir: str):
    """获取排序后的 wav 文件列表"""
    input_dir = Path(input_dir)
    files = list(input_dir.glob("*.wav"))

    def sort_key(p: Path):
        m = re.search(r"-(\d+)_(\d+)\.wav$", p.name, re.IGNORECASE)
        return (int(m.group(1)), int(m.group(2))) if m else (999, 999)

    return sorted(files, key=sort_key)


def get_audio_duration(file_path: str) -> float:
    """获取音频时长（秒）"""
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(file_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return float(result.stdout.strip())


def parse_srt_time(time_str: str) -> timedelta:
    """解析 SRT 时间戳字符串为 timedelta"""
    # SRT 格式: 00:00:00,000
    match = re.match(r"(\d{2}):(\d{2}):(\d{2}),(\d{3})", time_str.strip())
    if not match:
        raise ValueError(f"无效的时间戳格式: {time_str}")
    h, m, s, ms = map(int, match.groups())
    return timedelta(hours=h, minutes=m, seconds=s, milliseconds=ms)


def format_srt_time(td: timedelta) -> str:
    """将 timedelta 格式化为 SRT 时间戳"""
    total_seconds = td.total_seconds()
    if total_seconds < 0:
        total_seconds = 0
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = int(total_seconds % 60)
    milliseconds = int((total_seconds * 1000) % 1000)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}"


def merge_srt_files(srt_pairs, output_srt: str):
    """
    合并多个 SRT 文件，偏移时间戳

    Args:
        srt_pairs: list of (srt_path, time_offset_seconds)
        output_srt: 输出 SRT 路径
    """
    output_srt = Path(output_srt)
    output_srt.parent.mkdir(parents=True, exist_ok=True)

    global_idx = 0
    lines = []

    for srt_path, offset_sec in srt_pairs:
        srt_path = Path(srt_path)
        if not srt_path.exists():
            logger.warning(f"⚠️  SRT 不存在，跳过: {srt_path.name}")
            continue

        with open(srt_path, "r", encoding="utf-8") as f:
            content = f.read().strip()

        if not content:
            continue

        # 按空行分块
        blocks = re.split(r"\n\s*\n", content)

        for block in blocks:
            block_lines = block.strip().split("\n")
            if len(block_lines) < 3:
                continue

            # 第一行：序号（忽略，用全局序号）
            # 第二行：时间戳
            time_line = block_lines[1]
            match = re.match(r"(.+?)\s*-->\s*(.+)", time_line)
            if not match:
                continue

            start_str, end_str = match.groups()
            start = parse_srt_time(start_str) + timedelta(seconds=offset_sec)
            end = parse_srt_time(end_str) + timedelta(seconds=offset_sec)

            global_idx += 1
            lines.append(str(global_idx))
            lines.append(f"{format_srt_time(start)} --> {format_srt_time(end)}")
            lines.extend(block_lines[2:])  # 文本内容
            lines.append("")  # 空行分隔

    with open(output_srt, "w", encoding="utf-8") as f:
        f.write("\n".join(lines).strip() + "\n")

    logger.info(f"✅ SRT 合并完成: {output_srt.name} (共 {global_idx} 条)")


def concatenate_wavs(wav_files, output_wav: str):
    """使用 ffmpeg concat 拼接 wav 文件"""
    output_wav = Path(output_wav)
    output_wav.parent.mkdir(parents=True, exist_ok=True)

    if len(wav_files) == 1:
        # 单个文件直接复制
        subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-i", str(wav_files[0]), "-c", "copy", str(output_wav)],
            check=True
        )
        logger.info(f"✅ 单文件复制: {output_wav.name}")
        return

    # 生成 file list
    file_list = ""
    for wf in wav_files:
        file_list += f"file '{wf}'\n"

    file_list_path = output_wav.parent / "concat_list.txt"
    with open(file_list_path, "w") as f:
        f.write(file_list)

    cmd = [
        "ffmpeg", "-y", "-v", "error",
        "-f", "concat", "-safe", "0",
        "-i", str(file_list_path),
        "-c", "copy",
        str(output_wav)
    ]
    subprocess.run(cmd, check=True)
    file_list_path.unlink(missing_ok=True)
    logger.info(f"✅ 音频拼接完成: {output_wav.name} ({len(wav_files)} 个文件)")


def concat_and_merge(input_dir: str, output_dir: str = None):
    """
    扫描目录，按顺序拼接 wav 并合并 srt

    Args:
        input_dir: 输入目录（包含 wav/srt 对）
        output_dir: 输出目录（默认 input_dir 下的 output_merged）
    """
    input_dir = Path(input_dir)
    if not input_dir.exists():
        raise ValueError(f"目录不存在: {input_dir}")

    if output_dir is None:
        output_dir = input_dir / "output_merged"
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    new_file_prefix = Path(output_dir).stem
    wav_files = get_sorted_wav_files(input_dir)
    if not wav_files:
        logger.warning("⚠️  未找到 wav 文件")
        return

    # 构建对应的 srt 对
    srt_map = {}
    for wf in wav_files:
        srt_path = wf.with_suffix(".srt")
        if srt_path.exists():
            srt_map[wf.stem] = srt_path
        else:
            logger.warning(f"⚠️  缺少 SRT: {wf.stem}.srt")

    max_seconds = MAX_HOURS * 3600
    batch_idx = 1

    logger.info(f"📁 共 {len(wav_files)} 个音频文件，开始分段拼接（上限 {MAX_HOURS}h）...")

    # ========== 分批逻辑 ==========
    batches = []
    batch = []
    batch_duration = 0.0

    for wf in wav_files:
        dur = get_audio_duration(wf)

        # 加入当前文件后会超过上限，且 batch 非空 → 切分
        if batch and batch_duration + dur > max_seconds:
            batches.append(batch)
            batch = []
            batch_duration = 0.0

        batch.append(wf)
        batch_duration += dur

    if batch:
        batches.append(batch)

    logger.info(f"📦 共分为 {len(batches)} 个批次")

    # ========== 逐批处理 ==========
    for batch in batches:
        out_wav = output_dir / f"{new_file_prefix}-0_{batch_idx}.wav"
        out_srt = output_dir / f"{new_file_prefix}-0_{batch_idx}.srt"

        # 1. 拼接音频
        concatenate_wavs([str(f) for f in batch], str(out_wav))

        # 2. 合并 SRT（需要计算每个文件的时间偏移）
        srt_pairs = []
        time_offset = 0.0
        for wf in batch:
            srt_path = srt_map.get(wf.stem)
            if srt_path:
                srt_pairs.append((srt_path, time_offset))
            time_offset += get_audio_duration(wf)

        if srt_pairs:
            merge_srt_files(srt_pairs, str(out_srt))

        final_duration = get_audio_duration(out_wav)
        hours = final_duration / 3600
        logger.info(f"📦 批次 {batch_idx}: {len(batch)} 个文件, 总时长 {hours:.2f}h")

        batch_idx += 1

    logger.info(f"✅ 全部完成！输出目录: {output_dir}")

book_name = "zzqlyzgsh"
input_dir = f"/content/drive/MyDrive/{book_name}"
output_dir = f"/content/drive/MyDrive/{book_name}2"
# =============== 使用示例 ===============
concat_and_merge(
    input_dir=input_dir,
    output_dir=output_dir,
)
